# Notebook 00 — Data Preparation

This notebook downloads and preprocesses all datasets used in the MI-BCI pipeline.  
Each dataset is saved as a standardised `.npz` file to Google Drive so it persists across Colab sessions.

| Dataset | Source | Channels | Hz | Subjects | Classes |
|---|---|---|---|---|---|
| BCI IV 2a | MOABB (auto-download) | 22 | 250 | 9 | 4 |
| PhysioNet EEGMMIDB | MOABB (auto-download) | 64 | 160 | 109 | 4 |
| MIMED | Mendeley (auto-download) | 14 | 128 | 30 | 3 |

**Output:** `bciv2a.npz`, `physionet.npz`, `mimed.npz`  
Shape convention: `X (n_trials, n_channels, n_timepoints)`, `y (n_trials,)`

## 1. Environment Setup

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import os
PROJECT_PATH = '/content/drive/MyDrive/Honours Project'
os.makedirs(PROJECT_PATH, exist_ok=True)
os.chdir(PROJECT_PATH)

In [ ]:
!pip install moabb mne numpy scipy scikit-learn -q

In [3]:
import numpy as np
import mne
from scipy.signal import butter, lfilter
mne.set_log_level('WARNING')
print('Ready.')

Ready.


## 2. BCI Competition IV Dataset 2a

Auto-downloaded via MOABB — no manual steps required.

- 9 subjects, 22 EEG channels, 250Hz
- 4 classes: Left Hand (0), Right Hand (1), Feet (2), Tongue (3)
- 288 trials/subject → 5,184 trials total
- Epoched 0–4s after cue, bandpass filtered 8–30Hz (mu + beta band)

In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

dataset  = BNCI2014_001()
paradigm = MotorImagery(
    events=['left_hand', 'right_hand', 'feet', 'tongue'],
    n_classes=4,
    fmin=8, fmax=30,
    tmin=0, tmax=4
)

X_list, y_list = [], []
for subject in dataset.subject_list:
    X_s, y_s, _ = paradigm.get_data(dataset, subjects=[subject])
    X_list.append(X_s)
    y_list.append(y_s)

X_bciv2a = np.concatenate(X_list).astype(np.float32)
y_bciv2a = np.concatenate(y_list).astype(np.int64)
print(f'X: {X_bciv2a.shape} | y: {y_bciv2a.shape} | classes: {np.unique(y_bciv2a)}')

In [ ]:
np.savez(os.path.join(PROJECT_PATH, 'bciv2a.npz'), X=X_bciv2a, y=y_bciv2a)
print('Saved: bciv2a.npz')

## 3. PhysioNet EEGMMIDB

Auto-downloaded via MOABB.

- 109 subjects, 64 EEG channels, 160Hz
- ~9,800 trials total
- Optional — used as a larger pretraining source if BCI IV 2a alone is insufficient

> First run may take some time to download all subjects.

In [ ]:
from moabb.datasets import PhysionetMI

dataset_pn  = PhysionetMI()
paradigm_pn = MotorImagery(
    events=['left_hand', 'right_hand', 'feet', 'rest'],
    n_classes=4,
    fmin=8, fmax=30,
    tmin=0, tmax=4
)

X_list, y_list = [], []
for subject in dataset_pn.subject_list:
    try:
        X_s, y_s, _ = paradigm_pn.get_data(dataset_pn, subjects=[subject])
        X_list.append(X_s)
        y_list.append(y_s)
    except Exception as e:
        print(f'Subject {subject} skipped: {e}')

X_physionet = np.concatenate(X_list).astype(np.float32)
y_physionet = np.concatenate(y_list).astype(np.int64)
print(f'X: {X_physionet.shape} | y: {y_physionet.shape} | classes: {np.unique(y_physionet)}')

In [ ]:
np.savez(os.path.join(PROJECT_PATH, 'physionet.npz'), X=X_physionet, y=y_physionet)
print('Saved: physionet.npz')

## 4. MIMED Dataset

Downloaded directly from Mendeley Data via `wget`.

- 30 subjects, 14 EEG channels (EMOTIV Epoc X), 128Hz
- 3 classes: Left Hand (0), Right Hand (1), Stand (2)
- ~8 trials/subject per class
- Same hardware as our target deployment — most important dataset for fine-tuning

Each `.mat` file contains `joined_data`: cell array of 4 experiments × (n_timepoints, 14).  
Preprocessing: 3s baseline correction → bandpass 8–30Hz → z-score normalisation.

In [21]:
import os

# Download and extract MIMED from Mendeley
MIMED_ZIP = os.path.join(PROJECT_PATH, 'mimed_raw.zip')
MIMED_DIR = os.path.join(PROJECT_PATH, 'MIMED')

if not os.path.exists(MIMED_DIR):
    print('Downloading MIMED...')
    !wget -q --show-progress -O "{MIMED_ZIP}" "https://data.mendeley.com/public-api/zip/zs25xxjkm9/download/3"
    print('Extracting...')
    !unzip -q "{MIMED_ZIP}" -d "{PROJECT_PATH}"
    os.remove(MIMED_ZIP)
    MI_ZIP = os.path.join(MIMED_DIR, 'Motor Imagery.zip')
    !unzip -q "{MI_ZIP}" -d "{MIMED_DIR}"
    print('Done.')
else:
    print('MIMED already exists, skipping download.')

/content/drive/MyDr 100%[===================>]   1.10G  23.8MB/s    in 51s     
Extracting...
Done.


In [24]:
!ls -R MIMED/

MIMED/:
'1D_Data - Execution.py'  'Media Stimulus English Ver.mp4'
'1D_Data - Imagery.py'	  'Motor Execution.zip'
'Convert EDF-to-Mat.zip'  'Motor Imagery'
 Dataset_1D.zip		  'Motor Imagery.zip'
 DecisionTree_6Class.py    Suplementary_file.zip
'Matlab Dataset.zip'	   SVM_6Class.py

'MIMED/Motor Imagery':
'Left Hand Up-Down Imagine'   'Stand Up-Down Imagine'
'Right Hand Up-Down Imagine'

'MIMED/Motor Imagery/Left Hand Up-Down Imagine':
P01.mat  P05.mat  P09.mat  P13.mat  P17.mat  P21.mat  P25.mat  P29.mat
P02.mat  P06.mat  P10.mat  P14.mat  P18.mat  P22.mat  P26.mat  P30.mat
P03.mat  P07.mat  P11.mat  P15.mat  P19.mat  P23.mat  P27.mat
P04.mat  P08.mat  P12.mat  P16.mat  P20.mat  P24.mat  P28.mat

'MIMED/Motor Imagery/Right Hand Up-Down Imagine':
P01.mat  P05.mat  P09.mat  P13.mat  P17.mat  P21.mat  P25.mat  P29.mat
P02.mat  P06.mat  P10.mat  P14.mat  P18.mat  P22.mat  P26.mat  P30.mat
P03.mat  P07.mat  P11.mat  P15.mat  P19.mat  P23.mat  P27.mat
P04.mat  P08.mat  P12.mat  P16.mat  P20.

In [26]:
import scipy.io, glob

MIMED_PATH = os.path.join(PROJECT_PATH, 'MIMED/Motor Imagery')
FOLDERS = {
    0: 'Left Hand Up-Down Imagine',
    1: 'Right Hand Up-Down Imagine',
    2: 'Stand Up-Down Imagine'
}
FS, TRIAL_SEC = 128, 4
T = FS * TRIAL_SEC  # 512 timepoints

def bandpass(signal, lo=8, hi=30, fs=128, order=5):
    nyq  = fs / 2
    b, a = butter(order, [lo/nyq, hi/nyq], btype='band')
    return lfilter(b, a, signal, axis=0)

def load_subject(path, label):
    mat  = scipy.io.loadmat(path)
    data = mat['joined_data']
    X_trials, y_trials = [], []
    for exp in range(4):
        signal        = data[0, exp]                # (n_timepoints, 14)
        baseline_mean = signal[:384, :].mean(axis=0)
        trial         = signal[384:, :] - baseline_mean
        trial         = bandpass(trial)
        trial         = (trial - trial.mean(axis=0)) / (trial.std(axis=0) + 1e-6)
        n_trials      = trial.shape[0] // T
        for i in range(n_trials):
            seg = trial[i*T:(i+1)*T, :].T          # (14, 512)
            X_trials.append(seg)
            y_trials.append(label)
    return np.array(X_trials, dtype=np.float32), np.array(y_trials, dtype=np.int64)

X_list, y_list = [], []
for label, folder in FOLDERS.items():
    for fpath in sorted(glob.glob(f'{MIMED_PATH}/{folder}/*.mat')):
        X_s, y_s = load_subject(fpath, label)
        X_list.append(X_s)
        y_list.append(y_s)

X_mimed = np.concatenate(X_list)
y_mimed = np.concatenate(y_list)
print(f'X: {X_mimed.shape} | y: {y_mimed.shape} | classes: {np.unique(y_mimed)}')

X: (1080, 14, 512) | y: (1080,) | classes: [0 1 2]


In [34]:
np.savez(os.path.join(PROJECT_PATH, 'mimed.npz'), X=X_mimed, y=y_mimed)
print('Saved: mimed.npz')

Saved: mimed.npz


In [35]:
!ls

bciv2a.npz			   eegnet_pretrained_500.pth  results
eegnet_pretrained_500_new.pth	   MIMED
eegnet_pretrained_500_nonstop.pth  mimed.npz


## 5. Summary

Verify all `.npz` files before running downstream notebooks.

In [36]:
files = {
    'bciv2a.npz':    'BCI IV 2a',
    'physionet.npz': 'PhysioNet',
    'mimed.npz':     'MIMED',
}

print(f'{"File":<20} {"Dataset":<15} {"X shape":<25} {"Classes"}')
print('-' * 75)
for fname, name in files.items():
    path = os.path.join(PROJECT_PATH, fname)
    if os.path.exists(path):
        d = np.load(path, allow_pickle=True)
        print(f'{fname:<20} {name:<15} {str(d["X"].shape):<25} {np.unique(d["y"])}')
    else:
        print(f'{fname:<20} {name:<15} NOT FOUND — run section above first')

File                 Dataset         X shape                   Classes
---------------------------------------------------------------------------
bciv2a.npz           BCI IV 2a       (5184, 22, 1001)          [0 1 2 3]
physionet.npz        PhysioNet       NOT FOUND — run section above first
mimed.npz            MIMED           (1080, 14, 512)           [0 1 2]
